# Lab Work - 4.3

# Q1. The Splitting Criterion

In [ ]:
import numpy as np
import pandas as pd

x = np.array([1,2,3,4,5,6])
y = np.array([10,20,25,28,40,45])

df = pd.DataFrame({'x':x,'y':y})
df

## Candidate Split Points

In [ ]:
splits = [(x[i] + x[i+1])/2 for i in range(len(x)-1)]
print('Candidate Splits:', splits)

In [ ]:
results = []

for split in splits:
    left = y[x <= split]
    right = y[x > split]

    left_mean = np.mean(left)
    right_mean = np.mean(right)

    rss_left = np.sum((left-left_mean)**2)
    rss_right = np.sum((right-right_mean)**2)

    total_rss = rss_left + rss_right

    results.append([
        split,
        left_mean,
        right_mean,
        rss_left,
        rss_right,
        total_rss
    ])

table = pd.DataFrame(results,
                     columns=['Split','Left Mean','Right Mean','Left RSS','Right RSS','Total RSS'])

table

In [ ]:
best_row = table.loc[table['Total RSS'].idxmin()]

print('Best Split Found')
print(best_row)

The best threshold is the split producing the minimum Total RSS.

# Q2. Recursive Binary Splitting

In [ ]:
def rss(y_values):
    return np.sum((y_values - np.mean(y_values))**2)

def best_split(x_region,y_region):

    if len(x_region) < 2:
        return None

    candidates = []

    for i in range(len(x_region)-1):
        threshold = (x_region[i]+x_region[i+1])/2

        left = y_region[x_region <= threshold]
        right = y_region[x_region > threshold]

        total = rss(left)+rss(right)

        candidates.append((threshold,total))

    return min(candidates,key=lambda z:z[1])

In [ ]:
root_split = best_split(x,y)

print('Root Split')
print(root_split)

In [ ]:
root_threshold = root_split[0]

left_x = x[x<=root_threshold]
left_y = y[x<=root_threshold]

right_x = x[x>root_threshold]
right_y = y[x>root_threshold]

print('Left Region')
print(left_x,left_y)

print('Right Region')
print(right_x,right_y)

In [ ]:
left_split = best_split(left_x,left_y)
right_split = best_split(right_x,right_y)

print('Best Left Split:',left_split)
print('Best Right Split:',right_split)

Stopping Rule:

Stop splitting whenever a node contains fewer than 2 observations.

In [ ]:
print('Leaf Predictions')

print('Leaf 1 Mean =', np.mean(left_y[left_x<=left_split[0]]) if left_split else np.mean(left_y))
print('Leaf 2 Mean =', np.mean(left_y[left_x>left_split[0]]) if left_split else np.mean(left_y))
print('Leaf 3 Mean =', np.mean(right_y[right_x<=right_split[0]]) if right_split else np.mean(right_y))
print('Leaf 4 Mean =', np.mean(right_y[right_x>right_split[0]]) if right_split else np.mean(right_y))

### Final Tree Structure

```
Root
 ├── Left Region
 │     ├── Leaf 1
 │     └── Leaf 2
 └── Right Region
       ├── Leaf 3
       └── Leaf 4
```


# Q4. Predict and Compute Training RSS

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_leaf_nodes=4, random_state=42)
tree.fit(x.reshape(-1,1), y)

predictions = tree.predict(x.reshape(-1,1))

for xi, yi, pi in zip(x,y,predictions):
    print(f'x={xi}, Actual={yi}, Predicted={pi}')

In [ ]:
residuals = y - predictions

rss_tree = np.sum(residuals**2)

print('Tree RSS =', rss_tree)

In [ ]:
global_mean = np.mean(y)

rss_single = np.sum((y-global_mean)**2)

print('Single Node RSS =', rss_single)

print('Improvement =', rss_single-rss_tree)

### Reflection

If every leaf contains exactly one sample:

- Training RSS becomes 0.
- Test RSS generally increases.
- This is overfitting.

# Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(x,y,label='Actual Data')
plt.plot(x,predictions,linewidth=3,label='Tree Predictions')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Regression Tree Fit')
plt.legend()
plt.grid(True)
plt.show()

# Q4. Theory Questions

## Q4.1 Objective Function

At every node, a regression tree selects the split that minimizes:

RSS = Σ(yᵢ − ȳ_region)²

The best split is the one producing the smallest combined RSS across child nodes.

## Q4.2 Meaning of Leaf Prediction

Each leaf predicts the average response value of observations inside that leaf.

The mean minimizes Mean Squared Error (MSE), making it the optimal prediction.

## Q4.3 Regression Tree vs Linear Regression

| Regression Tree | Linear Regression |
|----------------|------------------|
| Piecewise constant model | Continuous linear model |
| Captures nonlinear relationships | Assumes linear relationship |
| Easy to interpret visually | Easy to interpret coefficients |
| Can overfit deeply | Can underfit nonlinear patterns |

## Q4.4 Why Global Optimization is NP-Hard

Finding the globally optimal tree requires evaluating an enormous number of possible split combinations.

The search space grows exponentially with dataset size.

Therefore practical algorithms use greedy recursive splitting:

- Choose the best split now.
- Recurse on child nodes.
- Sacrifice global optimality for computational efficiency.

# Conclusion

- Regression trees partition the feature space into regions.
- Each split minimizes RSS.
- Recursive binary splitting builds the tree greedily.
- Leaf predictions are region means.
- Deep trees reduce training RSS but risk overfitting.